In [1]:
# Enable autoreload when modifying files
%load_ext autoreload
%autoreload 2

In [3]:
import numpy as np
import pandas as pd

from icare_risk.clinphen import (
    DEFAULT_REGISTRY,
    DEFAULT_SCHEMA,
    DuckDBSource,
    EpisodeContext,
    FeatureMatrixBuilder,
)
from icare_risk.clinphen.phenotypes.historical import history_of_diabetes
from icare_risk.clinphen.phenotypes.windowed import min_spo2_24h
from icare_risk.clinphen.phenotypes.cross_domain import early_aki_risk

FAIL = []

def check(label, cond):
    status = "PASS" if cond else "FAIL"
    print(f"[{status}] {label}")
    if not cond:
        FAIL.append(label)

In [4]:
# ---------------------------------------------------------------------
# PART 2 — Full pipeline integration test through FeatureMatrixBuilder
# (physical column names on purpose -- proves schema decoupling works)
# ---------------------------------------------------------------------
print("\n=== PART 2: Full FeatureMatrixBuilder pipeline (DuckDB-stub) ===")

episodes_raw = pd.DataFrame({
    "SUBJECT": [101, 102],
    "SPELL_IDENTIFIER": ["SP-1", "SP-2"],
    "ENCNTR_ID": ["ENC-1", "ENC-2"],
    "ADMISSION_DATE": ["2024-03-10", "2024-03-15"],
    "ADMISSION_TIME": ["08:00:00", "14:00:00"],
    "DISCHARGE_DATE": ["2024-03-20", "2024-03-18"],
    "AGE_AT_ADMISSION": [67, 54],
    "MAIN_SPECIALTY_CODE": ["MED", "SURG"],
})

problems_raw = pd.DataFrame({
    "SUBJECT": [101],
    "PROBLEM_DT_TM": ["2020-01-01 10:00"],   # well before patient 101's admission
    "PROBLEM_CODE": ["E11.4"],
    "PROBLEM_DESC": ["Type 2 diabetes with neuro complication"],
    "ENCNTR_ID": ["ENC-0"],
})

prescribing_raw = pd.DataFrame({
    "SUBJECT": [101, 101, 102],
    "ORDER_DT_TM": ["2024-03-10 10:00", "2024-03-19 09:00", "2024-03-15 20:00"],
    "MEDICATION_NAME": ["Paracetamol", "Vancomycin Injection", "Metformin"],  # patient 102's Metformin is AFTER admission
    "MEDICATION_NAME_SHORT": ["Para", "Vanc", "Metf"],
    "THERAPEUTICAL_CLASS": ["Analgesic", "Antibiotic", "Antidiabetic"],
    "ORDERED_DOSE": ["1g", "1g", "500mg"],
    "ORDER_ID": ["O1", "O2", "O3"],
    "ENCNTR_ID": ["ENC-1", "ENC-1", "ENC-2"],
})

vitals_raw = pd.DataFrame({
    "SUBJECT": [101, 101, 102],
    "OBSERVATION_PERFORMED_DT": ["2024-03-10 09:00", "2024-03-10 22:00", "2024-03-15 15:00"],
    "OBSERVATION_CODE": ["SPO2", "SPO2", "SPO2"],
    "OBSERVATION_NAME": ["SpO2", "SpO2", "SpO2"],
    "OBSERVATION_RESULT_CLEAN": ["98", "89", "97"],
    "ENCNTR_ID": ["ENC-1", "ENC-1", "ENC-2"],
})

pathology_raw = pd.DataFrame({
    "SUBJECT": [101, 101, 102],
    "SAMPLE_COLLECTED_DT": ["2024-03-01 09:00", "2024-03-19 08:00", "2024-03-16 08:00"],
    "TEST_CODE": ["CREATININE", "CREATININE", "CREATININE"],
    "TEST_NAME": ["Creatinine", "Creatinine", "Creatinine"],
    "RESULT_CLEANED": ["1.0", "2.0", "0.8"],
    "RESULT_LOWER_RANGE": [0.6, 0.6, 0.6],
    "RESULT_UPPER_RANGE": [1.3, 1.3, 1.3],
    "PBAID": ["P1", "P2", "P3"],
})

con = __import__("duckdb").connect()
con.register("episodes_tbl", episodes_raw)
con.register("problems_tbl", problems_raw)
con.register("prescribing_tbl", prescribing_raw)
con.register("vitals_tbl", vitals_raw)
con.register("pathology_tbl", pathology_raw)


=== PART 2: Full FeatureMatrixBuilder pipeline (DuckDB-stub) ===


In [5]:
import dataclasses
schema = dataclasses.replace(
    DEFAULT_SCHEMA,
    episodes=dataclasses.replace(DEFAULT_SCHEMA.episodes, source="episodes_tbl"),
    domains={
        "problems": dataclasses.replace(DEFAULT_SCHEMA.domains["problems"], source="problems_tbl"),
        "prescribing": dataclasses.replace(DEFAULT_SCHEMA.domains["prescribing"], source="prescribing_tbl"),
        "vitals": dataclasses.replace(DEFAULT_SCHEMA.domains["vitals"], source="vitals_tbl"),
        "pathology": dataclasses.replace(DEFAULT_SCHEMA.domains["pathology"], source="pathology_tbl"),
        "microbiology": DEFAULT_SCHEMA.domains["microbiology"],
    },
)

builder = FeatureMatrixBuilder(schema=schema, source=DuckDBSource(connection=con))
matrix = builder.build(subjects=[101, 102], raise_on_error=True)

print(matrix.to_string())

row101 = matrix.loc[("SP-1", "ENC-1")]
row102 = matrix.loc[("SP-2", "ENC-2")]

check("Pipeline: patient 101 history_of_diabetes True (prior E11.4)", row101["history_of_diabetes"] == True)
check("Pipeline: patient 102 history_of_diabetes False (Metformin is POST-admission)", row102["history_of_diabetes"] == False)
check("Pipeline: patient 101 min_spo2_24h == 89 (both readings within 24h)", row101["min_spo2_24h"] == 89.0)
check("Pipeline: patient 102 min_spo2_24h == 97", row102["min_spo2_24h"] == 97.0)
check("Pipeline: patient 101 baseline_creatinine == 1.0 (historical, pre-admission)",
      row101["early_aki_risk__baseline_creatinine"] == 1.0)
check("Pipeline: patient 101 peak_creatinine_48h is NaN (only historical + far-future labs exist, none within 48h)",
      pd.isna(row101["early_aki_risk__peak_creatinine_48h"]))
check("Pipeline: patient 101 iv_vancomycin_48h False (Vancomycin ordered on day 9, outside 48h window)",
      row101["early_aki_risk__iv_vancomycin_48h"] == False)

# --- Cross-path consistency: same patient, same phenotype, both APIs agree ---
ctx_direct = EpisodeContext.from_frames(
    subject=101,
    index_admission=pd.Timestamp("2024-03-10 08:00:00"),
    frames={
        "vitals": pd.DataFrame({
            "timestamp": ["2024-03-10 09:00", "2024-03-10 22:00"],
            "code": ["SPO2", "SPO2"],
            "value": ["98", "89"],
        })
    },
)
check("Cross-path consistency: notebook EpisodeContext == full pipeline result for min_spo2_24h",
      min_spo2_24h(ctx_direct) == row101["min_spo2_24h"])

print(f"\n=== TOTAL FAILURES: {len(FAIL)} ===")
if FAIL:
    for f in FAIL:
        print(" -", f)
    raise SystemExit(1)
print("ALL CHECKS PASSED.")


                            early_aki_risk__baseline_creatinine  early_aki_risk__peak_creatinine_48h  early_aki_risk__delta_creatinine  early_aki_risk__iv_vancomycin_48h  early_aki_risk__aki_flag  history_of_diabetes  min_spo2_24h
SPELL_IDENTIFIER ENCNTR_ID                                                                                                                                                                                                            
SP-1             ENC-1                                      1.0                                  NaN                               NaN                              False                     False                 True          89.0
SP-2             ENC-2                                      NaN                                  0.8                               NaN                              False                     False                False          97.0
[PASS] Pipeline: patient 101 history_of_diabetes True (prior E11.4)
[PASS] P